# Stage B / NB 12 — Anatomy-aware mRALE decomposition, the E4 grid

Protocol reference: experiment family **E4**; research question **RQ4**; the load-bearing
evidence for the novelty claim challenged by referee 2a, and the answer to referee 1.2's
question about what image cropping contributes.

## What this notebook decides

Whether decomposing mRALE into per-lung regional predictions is a genuine inductive bias or
merely a reformatting of the same information. Seven arms, one fixed set of cached lung boxes,
identical predictor and prompts throughout:

| arm | view | prediction | isolates |
| --- | --- | --- | --- |
| E4a | V0 whole image | four components, total constrained to R+L | the baseline |
| E4b | V1 thorax crop | four components, total constrained to R+L | does cropping alone help? (referee 1.2's YOLOv8 question) |
| E4c | V0 + box coordinates as text | four components, total constrained to R+L | do coordinates alone help? |
| E4d | V2L + V2R masked views | per-lung, total constrained to R+L | **the anatomy-aware arm** |
| E4e | V2L + V2R masked views | per-lung, model's own reported total | does the arithmetic constraint matter? |
| E4f | ground-truth boxes | per-lung, constrained | oracle ceiling — how much does localization error cost? |
| E4g | heuristic PA boxes | per-lung, constrained | floor — is the learned localizer earning its keep? |

**E4c vs E4d is the load-bearing contrast.** If supplying coordinates as text matches supplying
separate regional views, then "anatomy-aware" reduces to prompt engineering and the claim should
be withdrawn. **E4f and E4g are what make a negative result interpretable** — they separate "the
idea is wrong" from "the localizer is not good enough".

## Read this before interpreting the output

NB 05 already ran the same decomposition on a frozen encoder and it **did not help**:
E0f (shared per-lung ordinal head on V2L/V2R) reached MAE 8.188 against E0d whole-image linear
8.057. That is one negative data point from a non-generative predictor. It does not settle E4 —
this grid uses the LoRA-adapted generative model, where the decomposition might compensate for a
hard counting task rather than supply a representational advantage — but it does mean a null
result here is a live possibility that must be reported rather than explained away.

Protocol risk K2 already commits to that: if E4d does not beat E4c beyond its confidence
interval, the anatomy-aware component is reported as a negative result with the E4g heuristic
floor. No new radiologist-scored boxes are available, so localization error cannot be separated
from decomposition error; this limitation must be explicit.

## Leakage discipline

Every internal number is **out-of-fold**: an image is scored only by the LoRA adapter from
`nb09_medgemma_lora/folds/fold_k/best_adapter` for the fold that held that image out.
Five-adapter ensembling is refused on internal images; four of five adapters trained on each of
them, so such an ensemble would be a leakage result. The gate enforces this.

## Efficiency

The adapter is loaded **once per fold** and all enabled arms are evaluated for that fold's held-out
images in a single pass. E4f is omitted when no eligible box annotations exist.

## Outputs (under `stage_B/nb12_anatomy_aware/`)
`anatomy_aware_predictions.jsonl` (per-item checkpointed), `e4_localization_ablation.csv`,
`per_fold_metrics.json`, per-arm `cross_fold_aggregate_95ci_<arm>.csv`, `e4_contrasts.csv`,
`regional_disagreement.csv`, `usability.json`, `gate_nb12.json`.

## 1. Imports, seeds, and the Stage A path contract

In [ ]:
import gc
import hashlib
import statistics
import json
import math
import os
import random
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions. Table 2 is only a valid comparison if every arm uses these.
_METRICS_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd() / "stage_B",
                   Path.cwd().parent / "stage_B"]
for _candidate in _METRICS_SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(
        "cxr_metrics.py not found. It must sit beside the Stage B notebooks; every arm in "
        f"Table 2 depends on its metric definitions. Searched: {_METRICS_SEARCH}")
import cxr_metrics as cm

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ---- Stage A path contract -------------------------------------------------------------
FALLBACK_STAGE_A_DIR = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
    Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json",
]
stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Path contract:", candidate)
        break
if stage_paths is None:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # matches the tested LoRA notebooks

print("Stage B output root:", STAGE_B_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| BF16:", torch.cuda.is_bf16_supported())

In [ ]:
import re
from PIL import Image
from peft import PeftModel
from transformers import (AutoModelForImageTextToText, AutoProcessor,
                          StoppingCriteria, StoppingCriteriaList)

Image.MAX_IMAGE_PIXELS = None
if not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()):
    raise RuntimeError("A BF16-capable CUDA GPU is required.")


def model_load_kwargs(dtype=torch.bfloat16, **extra):
    """`torch_dtype` was renamed to `dtype`; use whichever this transformers accepts."""
    import inspect
    from transformers import PreTrainedModel
    try:
        params = inspect.signature(PreTrainedModel.from_pretrained).parameters
        key = "dtype" if "dtype" in params else "torch_dtype"
    except Exception:
        key = "torch_dtype"
    return {key: dtype, **extra}


# evaluate_arm / print_arm_summary come from the shared
# loader block below, so every Stage B arm is scored by identical code.


## 2. Configuration

In [ ]:
NB12_DIR = STAGE_B_DIR / "nb12_anatomy_aware"
NB12_DIR.mkdir(parents=True, exist_ok=True)
NB09_DIR = STAGE_B_DIR / "nb09_medgemma_lora"

MODEL_ID = "google/medgemma-1.5-4b-it"
MODEL_REVISION = MODEL_REVISIONS.get(MODEL_ID)
AGENT_NAME = "A2_medgemma_lora"
FOLD_ADAPTER = "folds/fold_{fold}/best_adapter"

# Geometry, identical to NB 04's cache. Asserted rather than assumed: changing it here would
# silently change what the boxes mean.
COORDINATE_SCALE = 1000
LUNG_BOX_MARGIN_FRACTION = 0.04
HEURISTIC_BOXES = {"left": [450, 80, 1000, 970], "right": [0, 80, 550, 970]}

E4_ARMS = OrderedDict([
    ("E4a_direct",        {"view": "v0",       "mode": "whole",    "constrain": True}),
    ("E4b_thorax_crop",   {"view": "v1",       "mode": "whole",    "constrain": True}),
    ("E4c_anatomy_whole", {"view": "v0",       "mode": "coords",   "constrain": True}),
    ("E4d_anatomy_aware", {"view": "v2",       "mode": "regional", "constrain": True}),
    ("E4e_no_constraint", {"view": "v2",       "mode": "regional", "constrain": False}),
    ("E4f_oracle_boxes",  {"view": "v2_gt",    "mode": "regional", "constrain": True}),
    ("E4g_heuristic_box", {"view": "v2_heur",  "mode": "regional", "constrain": True}),
])
RUN_ARMS = ["E4a_direct", "E4b_thorax_crop", "E4c_anatomy_whole",
            "E4d_anatomy_aware", "E4e_no_constraint", "E4g_heuristic_box"]
# E4f needs ground-truth lung boxes on an annotated subset; see section 4.
GROUND_TRUTH_BOX_CSV = None      # set to a CSV with filename,left_box,right_box to enable E4f

MAX_NEW_TOKENS = 256
USE_JSON_STOP_CRITERIA = True    # E6-Q
FOLDS = [0, 1, 2, 3, 4]
MAX_IMAGES_PER_FOLD = None       # set 8 for a wiring check, then None

print("Arms to run:", RUN_ARMS)
print("E4f (oracle):", "enabled" if GROUND_TRUTH_BOX_CSV else "DISABLED — no ground-truth boxes")
print("Output:", NB12_DIR)

## 3. Prompts — carried over byte-for-byte

From the tested `anatomy_aware_mrale_medgemma15_5fold` notebook. The regional and
coordinate-conditioned prompts are the ones the earlier work validated; rewording them would make
this grid incomparable with it for no benefit.

In [ ]:
MRALE_SYSTEM_PROMPT = (
    "You are a radiology assistant trained to evaluate frontal chest X-rays using the modified "
    "Radiographic Assessment of Lung Edema (mRALE) framework. Assess each lung independently and "
    "return only valid JSON with exactly these keys in this order: extent_right, density_right, "
    "extent_left, density_left, extent_right_numerical, density_right_numerical, "
    "extent_left_numerical, density_left_numerical, mRALE Score. Extent is an integer 0-4, density "
    "is an integer 0-3, each lung score is extent multiplied by density, and the total is the sum."
)


DIRECT_USER_PROMPT = (
    "Predict the qualitative and numerical involvement and density scores for both lungs and the "
    "total mRALE Score."
)


def anatomy_aware_user_prompt(left_box, right_box):
    return (
        "Use the image and the anatomy localizer's normalized 0-1000 lung regions to assess each lung "
        f"separately. Patient left lung bbox={left_box}; patient right lung bbox={right_box}. "
        "Reason internally about opacity extent and density within each lung, then return only the "
        "required JSON and ensure the total equals the two regional products."
    )


def regional_user_prompt(side, box):
    return (
        f"This image preserves the original PA CXR canvas but only the localized patient {side} lung "
        f"region is visible (normalized bbox={box}). Estimate the {side}-lung extent and density. "
        "Return the complete required mRALE JSON schema; only the numerical fields for the visible "
        f"{side} lung will be used by the constrained reasoning stage."
    )


def multimodal_messages(system_prompt, user_prompt):
    return [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": user_prompt},
        ]},
    ]


def bounded_integer(value, lower, upper):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    if not math.isfinite(number) or not lower <= number <= upper:
        return None
    return int(math.floor(number + 0.5))


def median_integer(values):
    valid = [value for value in values if value is not None]
    return int(math.floor(statistics.median(valid) + 0.5)) if valid else None


print("Prompts carried over from the tested anatomy-aware notebook.")
print("  system :", MRALE_SYSTEM_PROMPT[:88], "...")
print("  direct :", DIRECT_USER_PROMPT[:88], "...")

## 4. Load the cached views and the fold assignment

In [ ]:
def load_view_index():
    path = NB04_DIR / "view_index.csv"
    if not path.is_file():
        raise FileNotFoundError(f"{path} not found. Run Stage A NB 04 first.")
    frame = pd.read_csv(path)
    print(f"view_index.csv: {len(frame):,} rows, cohorts={dict(Counter(frame['cohort']))}")
    return frame


def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level."
        )
    frame = pd.read_csv(path)
    print(f"midrc_folds_v2.csv: {len(frame):,} images, "
          f"{frame['group_id'].nunique():,} groups, folds={dict(sorted(Counter(frame['fold']).items()))}")
    return frame


def build_cohort_table():
    # One row per image: labels + fold + every cached view path. This is the single table
    # every Stage B notebook trains and predicts from.
    views = load_view_index()
    folds = load_folds()

    internal = folds.merge(
        views[views["cohort"] == "MIDRC"].drop(columns=["held_out_fold"], errors="ignore"),
        on="filename", how="inner", suffixes=("", "_view"),
    )
    if len(internal) != len(folds):
        missing = set(folds["filename"]) - set(internal["filename"])
        raise RuntimeError(
            f"{len(missing)} fold images have no NB 04 localization row (e.g. "
            f"{sorted(missing)[:5]}). Re-run NB 04 with MAX_IMAGES=None."
        )
    internal["mrale_right"] = (internal["extent_right_numerical"]
                               * internal["density_right_numerical"])
    internal["mrale_left"] = (internal["extent_left_numerical"]
                              * internal["density_left_numerical"])
    internal["is_external"] = False

    external_rows = []
    external_dir = NB03_DIR / "external_manifests"
    if external_dir.is_dir():
        for manifest_path in sorted(external_dir.glob("*_manifest.csv")):
            frame = pd.read_csv(manifest_path)
            if "status" in frame.columns:
                frame = frame[frame["status"] == "OK"]
            if not len(frame):
                continue
            cohort = str(frame["cohort"].iloc[0])
            merged = frame.merge(
                views[views["cohort"] == cohort][
                    ["filename", "v0_image", "v1_thorax_image", "v2_left_image",
                     "v2_right_image", "left_box", "right_box", "any_fallback"]
                ],
                on="filename", how="inner",
            )
            merged["fold"] = -1
            merged["is_external"] = True
            merged["group_id"] = "external::" + merged["filename"].astype(str)
            for column in ["mrale_total_annotated", "mrale_right", "mrale_left",
                           "extent_right_numerical", "density_right_numerical",
                           "extent_left_numerical", "density_left_numerical"]:
                if column not in merged.columns:
                    merged[column] = np.nan
            if "mrale_total" in merged.columns:
                merged["mrale_total_annotated"] = merged["mrale_total"]
            external_rows.append(merged)

    table = pd.concat([internal] + external_rows, ignore_index=True, sort=False)
    table["image_key"] = table.apply(
        lambda row: f"{row.get('cohort', 'MIDRC')}::{row['filename']}", axis=1)
    print()
    print(f"Cohort table: {len(table):,} rows "
          f"({int((~table['is_external']).sum()):,} internal, "
          f"{int(table['is_external'].sum()):,} external)")
    return table


def grouped_inner_split(subset, fraction, seed):
    # Group-aware inner validation split, same construction as the tested notebooks: whole
    # groups move together so the inner split cannot leak either.
    groups = sorted(subset["group_id"].astype(str).unique())
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_validation = max(1, round(len(groups) * fraction))
    validation_groups = set(groups[:n_validation])
    is_validation = subset["group_id"].astype(str).isin(validation_groups)
    train, validation = subset[~is_validation], subset[is_validation]
    assert not (set(train["group_id"]) & set(validation["group_id"]))
    return train, validation


def ground_truth_fields(row):
    def maybe_int(value):
        return None if value is None or (isinstance(value, float) and math.isnan(value)) else int(value)
    covid = row.get("covid_positive")
    if isinstance(covid, float) and math.isnan(covid):
        covid = None
    return {
        "gt_covid": covid if covid in {"Yes", "No"} else None,
        "gt_mrale_total": maybe_int(row.get("mrale_total_annotated")),
        "gt_mrale_right": maybe_int(row.get("mrale_right")),
        "gt_mrale_left": maybe_int(row.get("mrale_left")),
        "gt_extent_right": maybe_int(row.get("extent_right_numerical")),
        "gt_density_right": maybe_int(row.get("density_right_numerical")),
        "gt_extent_left": maybe_int(row.get("extent_left_numerical")),
        "gt_density_left": maybe_int(row.get("density_left_numerical")),
    }


def evaluate_arm(rows, label):
    # Single entry point for metrics, so every arm in Table 2 is scored identically.
    covid_rows = [row for row in rows if row.get("gt_covid") is not None]
    metrics = {"arm": label, "n_rows": len(rows)}
    if covid_rows:
        metrics["covid"] = cm.classification_metrics(
            [row["gt_covid"] for row in covid_rows],
            [row.get("covid_pred") for row in covid_rows],
            [row.get("covid_score") for row in covid_rows],
        )
    mrale_rows = [row for row in rows if row.get("gt_mrale_total") is not None]
    if mrale_rows:
        metrics["mrale"] = cm.mrale_metrics(mrale_rows)
    metrics["output"] = cm.localization_free_metrics(rows)
    return metrics


def print_arm_summary(metrics):
    covid = metrics.get("covid", {})
    mrale = metrics.get("mrale", {})
    print(f"  {metrics['arm']:<34} "
          f"AUROC={covid.get('auroc', float('nan')):.4f} "
          f"balAcc={covid.get('balanced_accuracy', float('nan')):.4f} "
          f"spec={covid.get('specificity', float('nan')):.4f} | "
          f"mRALE MAE={mrale.get('mae', float('nan')):.3f} "
          f"QWK={mrale.get('qwk', float('nan')):.4f} "
          f"cov={mrale.get('coverage', float('nan')):.3f}")

In [ ]:
view_index = pd.read_csv(NB04_DIR / "view_index.csv")
folds = pd.read_csv(FOLD_DEF_DIR / "midrc_folds_v2.csv")
midrc_views = view_index[view_index["cohort"] == "MIDRC"].drop(
    columns=["held_out_fold", "image_key"], errors="ignore")
work = folds.drop(columns=["image_key"], errors="ignore").merge(
    midrc_views,
    on="filename", how="inner")
if len(work) != len(folds):
    raise RuntimeError(f"{len(folds) - len(work)} fold images have no NB 04 localization row.")
work["image_key"] = "MIDRC::" + work["filename"].astype(str)
if work["image_key"].isna().any() or work["image_key"].duplicated().any():
    raise RuntimeError("Canonical image_key construction failed or produced duplicates.")

work["mrale_right"] = work["extent_right_numerical"] * work["density_right_numerical"]
work["mrale_left"] = work["extent_left_numerical"] * work["density_left_numerical"]
print(f"Internal images: {len(work):,}  folds={dict(sorted(Counter(work['fold']).items()))}")

fallback = float((work["any_fallback"] == True).mean())
laterality = float((work["laterality_plausible"] == True).mean())
print(f"Localization quality carried from NB 04: fallback={fallback:.4f} "
      f"laterality_plausible={laterality:.4f}")
print("  Report both alongside every E4 number: the anatomy-aware arm's credibility depends on")
print("  the localizer's MEASURED quality, not its assumed quality.")

# ---- E4f: ground-truth boxes -------------------------------------------------------------
def validated_box(value, label):
    if isinstance(value, str):
        value = json.loads(value)
    if not isinstance(value, (list, tuple)) or len(value) != 4:
        raise ValueError(f"{label} must contain [x1, y1, x2, y2].")
    box = [float(v) for v in value]
    if not all(math.isfinite(v) for v in box):
        raise ValueError(f"{label} contains a non-finite coordinate: {box}")
    x1, y1, x2, y2 = box
    if not (0 <= x1 < x2 <= COORDINATE_SCALE
            and 0 <= y1 < y2 <= COORDINATE_SCALE):
        raise ValueError(
            f"{label} is outside normalized 0-{COORDINATE_SCALE} coordinates: {box}")
    return box


ground_truth_boxes = {}
if GROUND_TRUTH_BOX_CSV and not Path(GROUND_TRUTH_BOX_CSV).is_file():
    raise FileNotFoundError(f"Configured E4f box CSV not found: {GROUND_TRUTH_BOX_CSV}")
if GROUND_TRUTH_BOX_CSV and Path(GROUND_TRUTH_BOX_CSV).is_file():
    box_frame = pd.read_csv(GROUND_TRUTH_BOX_CSV)
    required = {"filename", "left_box", "right_box"}
    if not required.issubset(box_frame.columns):
        raise ValueError(f"E4f CSV is missing columns: {sorted(required - set(box_frame.columns))}")
    for row_number, row in enumerate(box_frame.to_dict("records"), start=2):
        filename = str(row["filename"])
        if filename in ground_truth_boxes:
            raise ValueError(f"Duplicate E4f filename at CSV row {row_number}: {filename}")
        left = validated_box(row["left_box"], f"row {row_number} left_box")
        right = validated_box(row["right_box"], f"row {row_number} right_box")
        if (left[0] + left[2]) <= (right[0] + right[2]):
            raise ValueError(
                f"row {row_number}: patient-left box must lie to image-right of the "
                f"patient-right box for a frontal PA image ({filename}).")
        ground_truth_boxes[filename] = {"left": left, "right": right}
    print(f"\nE4f: {len(ground_truth_boxes):,} ground-truth boxes loaded.")
else:
    print()
    print("E4f (oracle ceiling) is DISABLED: no ground-truth lung boxes are available.")
    print("  Consequence, and it matters for how a null result can be reported: without the")
    print("  oracle bound, a failure of E4d cannot be attributed between 'the decomposition is")
    print("  not useful' and 'the localizer is not accurate enough'. Protocol risk K2 assumes")
    print("  E4f exists. No new radiologist-scored boxes are available, so report this as")
    print("  an unresolved localization-versus-decomposition limitation.")
    if "E4f_oracle_boxes" in RUN_ARMS:
        RUN_ARMS.remove("E4f_oracle_boxes")

## 5. View construction, generation, and parsing

`E4g` needs masked views built from the heuristic PA boxes rather than NB 04's learned boxes, so
those are generated here and cached. Everything else reads NB 04's cache.

In [ ]:
HEUR_DIR = NB12_DIR / "views_heuristic"
for side in ["left", "right"]:
    (HEUR_DIR / side).mkdir(parents=True, exist_ok=True)
    if ground_truth_boxes:
        (NB12_DIR / "views_gt" / side).mkdir(parents=True, exist_ok=True)


def expand_box(box, margin=LUNG_BOX_MARGIN_FRACTION):
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    return [max(0, x1 - margin * w), max(0, y1 - margin * h),
            min(COORDINATE_SCALE, x2 + margin * w), min(COORDINATE_SCALE, y2 + margin * h)]


def masked_view(image_path, normalized_box, output_path):
    """Preserves the canvas and blacks out everything outside the box — NOT a resized crop.
    Resizing a single lung destroys the PA laterality cue and the relative-size cue that
    extent judgement depends on. Identical construction to NB 04."""
    if Path(output_path).is_file():
        return str(output_path)
    with Image.open(image_path) as handle:
        image = handle.convert("RGB")
    width, height = image.size
    x1, y1, x2, y2 = expand_box(normalized_box)
    box = (max(0, round(x1 * width / COORDINATE_SCALE)),
           max(0, round(y1 * height / COORDINATE_SCALE)),
           min(width, round(x2 * width / COORDINATE_SCALE)),
           min(height, round(y2 * height / COORDINATE_SCALE)))
    canvas = Image.new("RGB", image.size, (0, 0, 0))
    canvas.paste(image.crop(box), box[:2])
    canvas.save(output_path)
    image.close()
    return str(output_path)


def parse_box(value):
    if isinstance(value, str):
        try: return json.loads(value)
        except Exception: return None
    return value


def views_for(record, arm_view):
    """Return the image path(s) an arm needs, plus the boxes it should quote."""
    left_box = parse_box(record.get("left_box"))
    right_box = parse_box(record.get("right_box"))
    if arm_view == "v0":
        return {"whole": record["v0_image"], "left_box": left_box, "right_box": right_box}
    if arm_view == "v1":
        path = record.get("v1_thorax_image")
        return {"whole": path if isinstance(path, str) and path else record["v0_image"],
                "left_box": left_box, "right_box": right_box}
    if arm_view == "v2":
        return {"left": record["v2_left_image"], "right": record["v2_right_image"],
                "left_box": left_box, "right_box": right_box}
    if arm_view == "v2_heur":
        stem = str(record["filename"]).replace("/", "_")
        return {"left": masked_view(record["v0_image"], HEURISTIC_BOXES["left"],
                                    HEUR_DIR / "left" / f"{stem}.png"),
                "right": masked_view(record["v0_image"], HEURISTIC_BOXES["right"],
                                     HEUR_DIR / "right" / f"{stem}.png"),
                "left_box": HEURISTIC_BOXES["left"], "right_box": HEURISTIC_BOXES["right"]}
    if arm_view == "v2_gt":
        boxes = ground_truth_boxes.get(str(record["filename"]))
        if boxes is None:
            return None
        stem = str(record["filename"]).replace("/", "_")
        return {"left": masked_view(record["v0_image"], boxes["left"],
                                    NB12_DIR / "views_gt" / "left" / f"{stem}.png"),
                "right": masked_view(record["v0_image"], boxes["right"],
                                     NB12_DIR / "views_gt" / "right" / f"{stem}.png"),
                "left_box": boxes["left"], "right_box": boxes["right"]}
    raise ValueError(arm_view)

In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID, **({"revision": MODEL_REVISION} if MODEL_REVISION else {}))
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"


class BalancedJsonStop(StoppingCriteria):
    """E6-Q: halt at the first complete top-level JSON object."""
    def __init__(self, tokenizer, prompt_length):
        self.tokenizer, self.prompt_length = tokenizer, prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        text = self.tokenizer.decode(input_ids[0, self.prompt_length:],
                                     skip_special_tokens=True)
        start = text.find("{")
        if start < 0:
            return False
        depth, in_string, escape = 0, False, False
        for ch in text[start:]:
            if escape: escape = False; continue
            if ch == "\\": escape = True; continue
            if ch == '"': in_string = not in_string; continue
            if in_string: continue
            if ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0: return True
        return False


def adapter_weight_path(adapter):
    candidates = [adapter / "adapter_model.safetensors", adapter / "adapter_model.bin"]
    weights = [path for path in candidates if path.is_file() and path.stat().st_size > 0]
    if not weights:
        raise FileNotFoundError(
            f"{adapter} has no non-empty adapter_model.safetensors or adapter_model.bin.")
    return weights[0]


def validate_fold_adapter(fold):
    adapter = NB09_DIR / FOLD_ADAPTER.format(fold=fold)
    config_path = adapter / "adapter_config.json"
    if not config_path.is_file():
        raise FileNotFoundError(
            f"{adapter} not found. NB 12 scores each image with the adapter from the fold that "
            "held it out, so NB 09 must have completed all five folds first.")
    config = json.loads(config_path.read_text(encoding="utf-8"))
    saved_base = config.get("base_model_name_or_path")
    if saved_base and saved_base != MODEL_ID:
        raise RuntimeError(
            f"Fold {fold} adapter targets {saved_base}, expected {MODEL_ID}.")
    return adapter, config_path, adapter_weight_path(adapter)


def load_fold_adapter(fold):
    adapter, _, _ = validate_fold_adapter(fold)
    base = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, **model_load_kwargs(torch.bfloat16, device_map="auto",
                                      low_cpu_mem_usage=True,
                                      **({"revision": MODEL_REVISION} if MODEL_REVISION else {})))
    base.config.use_cache = True
    return PeftModel.from_pretrained(base, str(adapter), is_trainable=False).eval()


def model_device(model):
    for p in model.parameters():
        if p.device.type not in {"meta", "cpu"}:
            return p.device
    return torch.device("cuda")


@torch.inference_mode()
def generate(model, image_path, user_prompt):
    prompt = processor.apply_chat_template(
        multimodal_messages(MRALE_SYSTEM_PROMPT, user_prompt),
        add_generation_prompt=True, tokenize=False)
    with Image.open(image_path) as handle:
        image = handle.convert("RGB")
        inputs = processor(text=prompt, images=image, return_tensors="pt")
    device = model_device(model)
    moved = {k: (v.to(device=device, dtype=torch.bfloat16) if v.is_floating_point()
                 else v.to(device)) for k, v in inputs.items()}
    n_prompt = moved["input_ids"].shape[-1]
    stopping = (StoppingCriteriaList([BalancedJsonStop(processor.tokenizer, n_prompt)])
                if USE_JSON_STOP_CRITERIA else None)
    out = model.generate(**moved, do_sample=False, max_new_tokens=MAX_NEW_TOKENS,
                         pad_token_id=processor.tokenizer.pad_token_id,
                         eos_token_id=processor.tokenizer.eos_token_id,
                         stopping_criteria=stopping, use_cache=True)
    return processor.decode(out[0, n_prompt:], skip_special_tokens=True).strip()


def extract_json_object(text):
    cleaned = re.sub(r"^```(?:json)?\s*", "", (text or "").strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    if start < 0:
        raise ValueError("No JSON object found")
    obj, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    if not isinstance(obj, dict):
        raise ValueError("Not a JSON object")
    return obj


FIELDS = {"extent_right_numerical": 4, "density_right_numerical": 3,
          "extent_left_numerical": 4, "density_left_numerical": 3}


def parse_components(text):
    try:
        parsed = extract_json_object(text)
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"
    values = {}
    for field, upper in FIELDS.items():
        v = bounded_integer(parsed.get(field), 0, upper)
        if v is None:
            return None, f"missing_or_out_of_range:{field}"
        values[field] = v
    reported = bounded_integer(parsed.get("mRALE Score"), 0, 24)
    return {"values": values, "reported_total": reported}, None

## 6. Run the grid — one adapter load per fold, all arms in a single pass

Checkpointed per `(arm, image_key)`. Loading a 4B model once per fold rather than once per
(fold, arm) is the difference between ~25 and ~175 GPU-hours.

In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Normalize the key here as well so Section 6 can be rerun safely in a kernel that still
# holds a pre-fix `work` table with pandas-generated image_key_x/image_key_y columns.
if "image_key" not in work.columns:
    if "filename" not in work.columns:
        raise KeyError("NB 12 work table has neither image_key nor filename. Rerun Section 4.")
    work = work.copy()
    work["image_key"] = "MIDRC::" + work["filename"].astype(str)
    print("Reconstructed canonical image_key after the view/fold merge.")
if work["image_key"].isna().any() or work["image_key"].duplicated().any():
    raise RuntimeError("work.image_key must be non-null and unique before the E4 grid.")


adapter_manifest = {}
for fold in FOLDS:
    adapter_dir, config_path, weight_path = validate_fold_adapter(fold)
    adapter_manifest[str(fold)] = {
        "adapter_dir": str(adapter_dir),
        "config_sha256": sha256_file(config_path),
        "weights_file": weight_path.name,
        "weights_bytes": weight_path.stat().st_size,
        "weights_sha256": sha256_file(weight_path),
    }

cache_identity = {
    "schema_version": 2,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "folds": FOLDS, "max_images_per_fold": MAX_IMAGES_PER_FOLD,
    "arms": {arm: E4_ARMS[arm] for arm in RUN_ARMS},
    "coordinate_scale": COORDINATE_SCALE,
    "margin": LUNG_BOX_MARGIN_FRACTION, "heuristic_boxes": HEURISTIC_BOXES,
    "system_prompt": MRALE_SYSTEM_PROMPT, "direct_prompt": DIRECT_USER_PROMPT,
    "max_new_tokens": MAX_NEW_TOKENS,
    "json_stop_criteria": USE_JSON_STOP_CRITERIA,
    "fold_csv_sha256": sha256_file(FOLD_DEF_DIR / "midrc_folds_v2.csv"),
    "view_index_sha256": sha256_file(NB04_DIR / "view_index.csv"),
    "ground_truth_box_csv_sha256": (sha256_file(GROUND_TRUTH_BOX_CSV)
        if GROUND_TRUTH_BOX_CSV and Path(GROUND_TRUTH_BOX_CSV).is_file() else None),
    "adapters": adapter_manifest,
}
cache_fingerprint = hashlib.sha256(
    json.dumps(cache_identity, sort_keys=True).encode("utf-8")).hexdigest()

PRED_PATH = NB12_DIR / "anatomy_aware_predictions.jsonl"
FINGERPRINT_PATH = NB12_DIR / "anatomy_aware_predictions.fingerprint.json"
if PRED_PATH.is_file():
    if not FINGERPRINT_PATH.is_file():
        raise RuntimeError(
            f"{PRED_PATH} predates cache fingerprinting. Archive or delete it before rerun.")
    saved_fingerprint = json.loads(FINGERPRINT_PATH.read_text(encoding="utf-8"))
    if saved_fingerprint.get("fingerprint") != cache_fingerprint:
        raise RuntimeError(
            "NB 12 inputs differ from the cached prediction run. Archive or delete "
            f"{PRED_PATH} and {FINGERPRINT_PATH}; stale results will not be reused.")
else:
    cm.write_json(FINGERPRINT_PATH, {"fingerprint": cache_fingerprint,
                                     "identity": cache_identity})

done = cm.load_jsonl_by_key(PRED_PATH, ["arm", "image_key"])
print(f"Resuming with {len(done):,} fingerprint-matched (arm, image) results.")

records = work.to_dict("records")
by_fold = defaultdict(list)
for record in records:
    by_fold[int(record["fold"])].append(record)


def regional_prediction(model, record, spec, arm):
    views = views_for(record, spec["view"])
    if views is None:
        return None, "no_ground_truth_box"
    out, errors = {}, []
    for side, key in [("right", "right"), ("left", "left")]:
        box = views[f"{side}_box"]
        text = generate(model, views[key], regional_user_prompt(side, box))
        parsed, error = parse_components(text)
        if error:
            errors.append(f"{side}:{error}")
            continue
        out[side] = {
            "extent": parsed["values"][f"extent_{side}_numerical"],
            "density": parsed["values"][f"density_{side}_numerical"],
            "reported_total": parsed["reported_total"], "raw": text[:800]}
    if len(out) < 2:
        return None, "; ".join(errors) or "regional_parse_failed"
    right = out["right"]["extent"] * out["right"]["density"]
    left = out["left"]["extent"] * out["left"]["density"]
    if spec["constrain"]:
        total = right + left
    else:
        # E4e: trust whichever total the model reported, averaging the two regional views.
        reported = [out[s]["reported_total"] for s in ("right", "left")
                    if out[s]["reported_total"] is not None]
        total = int(round(sum(reported) / len(reported))) if reported else right + left
    return {"extent_right": out["right"]["extent"], "density_right": out["right"]["density"],
            "extent_left": out["left"]["extent"], "density_left": out["left"]["density"],
            "mrale_right": right, "mrale_left": left, "mrale_total": total,
            "raw": out["right"]["raw"]}, None


def whole_prediction(model, record, spec):
    views = views_for(record, spec["view"])
    prompt = (anatomy_aware_user_prompt(views["left_box"], views["right_box"])
              if spec["mode"] == "coords" else DIRECT_USER_PROMPT)
    text = generate(model, views["whole"], prompt)
    parsed, error = parse_components(text)
    if error:
        return None, error
    v = parsed["values"]
    right = v["extent_right_numerical"] * v["density_right_numerical"]
    left = v["extent_left_numerical"] * v["density_left_numerical"]
    return {"extent_right": v["extent_right_numerical"],
            "density_right": v["density_right_numerical"],
            "extent_left": v["extent_left_numerical"],
            "density_left": v["density_left_numerical"],
            "mrale_right": right, "mrale_left": left,
            "mrale_total": right + left if spec["constrain"]
                           else (parsed["reported_total"] if parsed["reported_total"] is not None
                                 else right + left),
            "raw": text[:800]}, None


for fold in FOLDS:
    fold_records = by_fold[fold]
    if MAX_IMAGES_PER_FOLD:
        fold_records = fold_records[:MAX_IMAGES_PER_FOLD]
    pending = [(arm, r) for arm in RUN_ARMS for r in fold_records
               if not (arm == "E4f_oracle_boxes"
                       and str(r["filename"]) not in ground_truth_boxes)
               and (arm, str(r["image_key"])) not in done]
    if not pending:
        print(f"fold {fold}: complete, skipping adapter load.")
        continue

    print("=" * 78)
    print(f"FOLD {fold}: {len(fold_records):,} held-out images, {len(pending):,} pending")
    model = load_fold_adapter(fold)
    started = time.perf_counter()
    for position, (arm, record) in enumerate(pending, start=1):
        spec = E4_ARMS[arm]
        t0 = time.perf_counter()
        try:
            if spec["mode"] == "regional":
                decoded, error = regional_prediction(model, record, spec, arm)
            else:
                decoded, error = whole_prediction(model, record, spec)
        except Exception as exc:
            decoded, error = None, f"generation_failed: {type(exc).__name__}: {exc}"
        row = cm.make_prediction_row(
            image_key=str(record["image_key"]), cohort="MIDRC", subcohort="MIDRC",
            filename=record["filename"], held_out_fold=fold,
            agent=AGENT_NAME, arm=arm, view=spec["view"], task="mrale_prediction",
            gt_mrale_total=int(record["mrale_total_annotated"]),
            gt_mrale_right=int(record["mrale_right"]), gt_mrale_left=int(record["mrale_left"]),
            gt_extent_right=int(record["extent_right_numerical"]),
            gt_density_right=int(record["density_right_numerical"]),
            gt_extent_left=int(record["extent_left_numerical"]),
            gt_density_left=int(record["density_left_numerical"]),
            gt_covid=record.get("covid_positive"),
            valid=decoded is not None, parse_error=error,
            seconds=round(time.perf_counter() - t0, 3),
            model_id=MODEL_ID, model_revision=MODEL_REVISION,
            any_fallback=bool(record.get("any_fallback")),
            laterality_plausible=bool(record.get("laterality_plausible")),
            **({k: v for k, v in decoded.items() if k != "raw"} if decoded else {}),
            raw_output=(decoded or {}).get("raw"))
        cm.append_jsonl(PRED_PATH, row)
        done[(arm, str(record["image_key"]))] = row
        if position % 100 == 0:
            rate = position / max(time.perf_counter() - started, 1e-6)
            print(f"    [{position}/{len(pending)}] {rate:.2f} pred/s "
                  f"~{(len(pending) - position) / max(rate, 1e-9) / 60:.0f} min left")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  fold {fold} done in {(time.perf_counter() - started) / 60:.1f} min")

predictions = list(done.values())
print()
print(f"Total predictions: {len(predictions):,}")

## 7. The E4 table and the contrasts that decide the claim

Each arm is scored pooled out-of-fold and per fold. The three contrasts that matter are computed
explicitly with paired, study-group-clustered bootstrap confidence intervals, because "E4d has a lower MAE" is not the
same claim as "E4d beats E4c beyond noise on the same images".

In [ ]:
rows_by_arm = defaultdict(list)
for row in predictions:
    rows_by_arm[row["arm"]].append(row)

per_fold_metrics, table_rows = {}, []
for arm in RUN_ARMS:
    arm_rows = rows_by_arm.get(arm, [])
    if not arm_rows:
        continue
    pooled = evaluate_arm(arm_rows, arm)
    per_fold = {}
    for fold in FOLDS:
        fold_rows = [r for r in arm_rows if r.get("held_out_fold") == fold]
        if fold_rows:
            per_fold[fold] = evaluate_arm(fold_rows, f"{arm}/fold{fold}")
    per_fold_metrics[arm] = per_fold
    aggregate = cm.aggregate_over_folds(per_fold) if per_fold else []
    if aggregate:
        pd.DataFrame(aggregate).to_csv(
            NB12_DIR / f"cross_fold_aggregate_95ci_{arm}.csv", index=False)

    def ci(metric):
        hit = [r for r in aggregate if r["metric"] == metric]
        return (f"[{hit[0]['ci95_lower']:.3f}, {hit[0]['ci95_upper']:.3f}]" if hit else None)

    m = pooled.get("mrale", {})
    table_rows.append(OrderedDict([
        ("arm", arm), ("view", E4_ARMS[arm]["view"]), ("mode", E4_ARMS[arm]["mode"]),
        ("constrained", E4_ARMS[arm]["constrain"]),
        ("n", pooled["n_rows"]),
        ("mae", round(m.get("mae", float("nan")), 3)),
        ("mae_ci95", ci("mrale.mae")),
        ("rmse", round(m.get("rmse", float("nan")), 3)),
        ("qwk", round(m.get("qwk", float("nan")), 4)),
        ("spearman", round(m.get("spearman_rho", float("nan")), 4)),
        ("within1", round(m.get("within1_accuracy", float("nan")), 4)),
        ("coverage", round(m.get("coverage", float("nan")), 4)),
        ("formula_consistency", round(m.get("formula_consistency", float("nan")), 4)),
        ("mae_right", round(m.get("mae_right", float("nan")), 3)),
        ("mae_left", round(m.get("mae_left", float("nan")), 3)),
        ("mae_band_none", round(m.get("mae_band_none", float("nan")), 3)),
        ("mae_band_mild", round(m.get("mae_band_mild", float("nan")), 3)),
        ("mae_band_moderate", round(m.get("mae_band_moderate", float("nan")), 3)),
        ("mae_band_severe", round(m.get("mae_band_severe", float("nan")), 3)),
    ]))
    print_arm_summary(pooled)

e4_table = pd.DataFrame(table_rows)
e4_table.to_csv(NB12_DIR / "e4_localization_ablation.csv", index=False)
cm.write_json(NB12_DIR / "per_fold_metrics.json", per_fold_metrics)
pd.set_option("display.width", 220)
print()
print(e4_table[["arm", "mae", "mae_ci95", "qwk", "within1", "coverage",
                "formula_consistency"]].to_string(index=False))

In [ ]:
# ---- Paired contrasts on the images both arms scored -------------------------------------
group_by_image = {str(row["image_key"]): str(row["group_id"])
                  for row in work.to_dict("records")}


def paired_delta(arm_a, arm_b, n_boot=2000, seed=SEED):
    """MAE(a)-MAE(b), cluster-bootstrapped over Stage A study groups."""
    a = {r["image_key"]: r for r in rows_by_arm.get(arm_a, [])}
    b = {r["image_key"]: r for r in rows_by_arm.get(arm_b, [])}
    shared = sorted(set(a) & set(b))
    if len(shared) < 30:
        return None
    def err(row):
        if row.get("mrale_total") is None:
            return cm.INVALID_TOTAL_PENALTY
        return abs(float(row["mrale_total"]) - float(row["gt_mrale_total"]))
    ea = np.array([err(a[k]) for k in shared])
    eb = np.array([err(b[k]) for k in shared])
    observed = float(ea.mean() - eb.mean())
    clusters = defaultdict(list)
    for index, image_key in enumerate(shared):
        clusters[group_by_image.get(str(image_key), str(image_key))].append(index)
    cluster_ids = sorted(clusters)
    if len(cluster_ids) < 10:
        return None
    rng = np.random.default_rng(seed)
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        sampled_clusters = rng.choice(cluster_ids, size=len(cluster_ids), replace=True)
        idx = np.concatenate([np.asarray(clusters[group], dtype=int)
                              for group in sampled_clusters])
        deltas[i] = ea[idx].mean() - eb[idx].mean()
    low, high = np.percentile(deltas, [2.5, 97.5])
    return {"contrast": f"{arm_a} - {arm_b}", "n_paired": len(shared),
            "n_groups": len(cluster_ids), "bootstrap_unit": "group_id",
            "delta_mae": round(observed, 4),
            "ci95_lower": round(float(low), 4), "ci95_upper": round(float(high), 4),
            "excludes_zero": bool(low > 0 or high < 0),
            "favours": (arm_b if observed > 0 else arm_a) if (low > 0 or high < 0) else "neither"}


CONTRASTS = [
    ("E4d_anatomy_aware", "E4c_anatomy_whole"),   # THE load-bearing test
    ("E4d_anatomy_aware", "E4a_direct"),          # decomposition vs baseline
    ("E4b_thorax_crop",   "E4a_direct"),          # does cropping alone help? (referee 1.2)
    ("E4e_no_constraint", "E4d_anatomy_aware"),   # does the arithmetic constraint matter?
    ("E4g_heuristic_box", "E4d_anatomy_aware"),   # is the learned localizer earning its keep?
    ("E4f_oracle_boxes",  "E4d_anatomy_aware"),   # what does localization error cost?
]
contrast_rows = [r for r in (paired_delta(a, b) for a, b in CONTRASTS) if r]
contrasts = pd.DataFrame(contrast_rows)
if len(contrasts):
    contrasts.to_csv(NB12_DIR / "e4_contrasts.csv", index=False)
    print("Paired contrasts (negative delta favours the FIRST arm):")
    print(contrasts.to_string(index=False))

print()
print("=" * 78)
print("VERDICT ON THE NOVELTY CLAIM")
key = next((r for r in contrast_rows
            if r["contrast"] == "E4d_anatomy_aware - E4c_anatomy_whole"), None)
if key is None:
    print("  E4c vs E4d could not be computed; the claim is untested.")
elif key["excludes_zero"] and key["delta_mae"] < 0:
    print(f"  E4d beats E4c by {abs(key['delta_mae']):.3f} MAE, CI "
          f"[{key['ci95_lower']}, {key['ci95_upper']}] excluding zero.")
    print("  Separate regional VIEWS beat coordinates-as-text. The anatomy-aware decomposition")
    print("  is a genuine inductive bias and the claim in protocol Section 6.2 E4 is supported.")
else:
    print(f"  E4d vs E4c delta {key['delta_mae']:+.3f} MAE, CI "
          f"[{key['ci95_lower']}, {key['ci95_upper']}] — does NOT exclude zero.")
    print("  Supplying coordinates as text matches supplying separate regional views, so")
    print("  'anatomy-aware' reduces to prompt content rather than a representational")
    print("  advantage. Protocol risk K2 applies: report this as a NEGATIVE result, quote the")
    print("  E4g floor, and move the paper's framing to E3/E7.")
    print("  NB 05's E0f reached the same conclusion on a frozen encoder (8.188 vs 8.057), so")
    print("  two independent predictors now agree. That is a finding, not a disappointment.")
    if not ground_truth_boxes:
        print()
        print("  WITHOUT E4f you cannot separate 'the idea is wrong' from 'the localizer is not")
        print("  good enough'. No new radiologist-scored boxes are available; disclose this")
        print("  unresolved localization-versus-decomposition limitation.")

## 8. Localization quality and regional-versus-whole disagreement

Required by the protocol alongside every E4 number: the anatomy-aware arm's credibility rests on
the localizer's measured quality. This section also reports where the regional and whole-image
arms disagree, which is the natural place to look for E4's failure modes.

In [ ]:
disagreement_rows = []
whole = {r["image_key"]: r for r in rows_by_arm.get("E4a_direct", [])}
regional = {r["image_key"]: r for r in rows_by_arm.get("E4d_anatomy_aware", [])}
for key in sorted(set(whole) & set(regional)):
    w, r = whole[key], regional[key]
    if w.get("mrale_total") is None or r.get("mrale_total") is None:
        continue
    gt = float(w["gt_mrale_total"])
    disagreement_rows.append({
        "image_key": key, "held_out_fold": w.get("held_out_fold"),
        "gt_mrale_total": gt, "severity_band": cm.severity_band(gt),
        "whole_total": w["mrale_total"], "regional_total": r["mrale_total"],
        "abs_disagreement": abs(w["mrale_total"] - r["mrale_total"]),
        "whole_error": abs(w["mrale_total"] - gt), "regional_error": abs(r["mrale_total"] - gt),
        "regional_better": abs(r["mrale_total"] - gt) < abs(w["mrale_total"] - gt),
        "any_fallback": r.get("any_fallback"),
        "laterality_plausible": r.get("laterality_plausible"),
    })
disagree = pd.DataFrame(disagreement_rows)
if len(disagree):
    disagree.to_csv(NB12_DIR / "regional_disagreement.csv", index=False)
    print(f"Paired whole-vs-regional images: {len(disagree):,}")
    print(f"  mean |disagreement|      : {disagree['abs_disagreement'].mean():.3f} mRALE points")
    print(f"  regional closer to truth : {disagree['regional_better'].mean():.1%}")
    print()
    print("  By severity band:")
    band = disagree.groupby("severity_band").agg(
        n=("image_key", "size"), mean_disagreement=("abs_disagreement", "mean"),
        regional_better=("regional_better", "mean"),
        whole_err=("whole_error", "mean"), regional_err=("regional_error", "mean"))
    print(band.round(3).to_string())
    print()
    print("  A regional-better rate near 50% means the two arms are exchangeable and the")
    print("  decomposition is not adding information, even if their pooled MAEs differ.")

    # Does localization quality predict where the regional arm wins?
    if disagree["any_fallback"].notna().any() and disagree["any_fallback"].nunique() > 1:
        print()
        print("  Regional-better rate by localization quality:")
        print(disagree.groupby("any_fallback")["regional_better"].agg(["mean", "size"])
              .round(4).to_string())

localization = {
    "fallback_rate": float((work["any_fallback"] == True).mean()),
    "laterality_plausible_rate": float((work["laterality_plausible"] == True).mean()),
    "source": str(NB04_DIR / "localization_metrics.csv"),
    "note": ("Report these beside every E4 number. A near-zero fallback rate means E4g's "
             "heuristic-box floor, not fallback contamination, is what bounds the learned "
             "localizer's contribution."),
}
cm.write_json(NB12_DIR / "localization_quality.json", localization)
print()
print("Localization quality carried from NB 04:", json.dumps(localization, indent=2)[:260])

## 9. Gate

In [ ]:
failures, warnings = [], []
usability = {}

expected = {str(k) for k in work["image_key"]}
oracle_expected = {str(row["image_key"]) for row in work.to_dict("records")
                   if str(row["filename"]) in ground_truth_boxes}
for arm in RUN_ARMS:
    arm_rows = rows_by_arm.get(arm, [])
    covered = {r["image_key"] for r in arm_rows}
    arm_expected = oracle_expected if arm == "E4f_oracle_boxes" else expected
    if MAX_IMAGES_PER_FOLD is None and covered != arm_expected:
        failures.append(
            f"{arm}: out-of-fold coverage {len(covered)} of {len(arm_expected)} eligible images.")
    repeated = [k for k, n in Counter(r["image_key"] for r in arm_rows).items() if n > 1]
    if repeated:
        failures.append(f"{arm}: {len(repeated)} images predicted more than once.")
    # Leakage: an image must be scored by the adapter of the fold that held it OUT.
    fold_by_key = {str(r["image_key"]): int(r["fold"]) for r in records}
    mismatched = [r["image_key"] for r in arm_rows
                  if r.get("held_out_fold") is not None
                  and fold_by_key.get(r["image_key"]) != r["held_out_fold"]]
    if mismatched:
        failures.append(
            f"{arm}: {len(mismatched)} images scored by an adapter from a fold that did NOT "
            "hold them out. That is a leakage result and invalidates the arm.")
    metrics = evaluate_arm(arm_rows, arm) if arm_rows else {}
    coverage = metrics.get("mrale", {}).get("coverage", float("nan"))
    usability[arm] = {"mrale_usable": bool(not math.isnan(coverage) and coverage > 0.0),
                      "mrale_coverage": None if math.isnan(coverage) else round(coverage, 4),
                      "score_usable": False}
    if not math.isnan(coverage) and coverage < 0.9:
        warnings.append(f"{arm}: coverage {coverage:.3f}; the penalised MAE is partly the "
                        "24-point invalid penalty. Report coverage beside every MAE.")

if "E4f_oracle_boxes" not in RUN_ARMS:
    warnings.append(
        "E4f (oracle ceiling) was NOT run — no ground-truth lung boxes exist. A null E4 result "
        "therefore cannot be attributed between 'the decomposition does not help' and 'the "
        "localizer is not accurate enough'. No new radiologist-scored boxes are available, "
        "so this limitation must be reported explicitly."
    )

if key is not None and not key["excludes_zero"]:
    warnings.append(
        f"NOVELTY CLAIM NOT SUPPORTED: E4d vs E4c delta {key['delta_mae']:+.3f} MAE with CI "
        f"[{key['ci95_lower']}, {key['ci95_upper']}] spanning zero. Protocol risk K2 applies. "
        "Report the negative result with the E4g floor and reframe around E3/E7 rather than "
        "presenting the pooled MAE ordering as if it were a difference."
    )

if len(disagree) and abs(float(disagree["regional_better"].mean()) - 0.5) < 0.03:
    warnings.append(
        f"Regional arm is closer to truth on {disagree['regional_better'].mean():.1%} of paired "
        "images — indistinguishable from a coin flip. The two views are exchangeable at the "
        "image level whatever the pooled MAEs say.")

cm.write_json(NB12_DIR / "usability.json", usability)
cm.write_json(NB12_DIR / "run_config.json", {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "12_anatomy_aware_mrale_5fold.ipynb",
    "protocol_experiments": list(E4_ARMS),
    "arms_run": RUN_ARMS,
    "evaluation_mode": "out_of_fold",
    "adapter_source": str(NB09_DIR / FOLD_ADAPTER),
    "geometry": {"coordinate_scale": COORDINATE_SCALE,
                 "margin": LUNG_BOX_MARGIN_FRACTION, "heuristic_boxes": HEURISTIC_BOXES},
    "decoding": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS,
                 "json_stop_criteria": USE_JSON_STOP_CRITERIA},
    "localization_quality": localization,
    "e4f_enabled": bool(ground_truth_boxes),
    "e4f_annotated_images": len(oracle_expected),
    "cache_fingerprint": cache_fingerprint,
    "adapter_manifest": adapter_manifest,
    "contrasts": contrast_rows,
    "prior_evidence": ("NB 05 E0f applied the same decomposition to a frozen encoder and did "
                       "not improve on the whole-image probe (8.188 vs 8.057)."),
})


def report(title, messages):
    print(title)
    for m in messages or []:
        print("  -", m)
    if not messages:
        print("  none")


report("WARNINGS", warnings)
print()
report("FAILURES", failures)
cm.write_json(NB12_DIR / "gate_nb12.json",
              {"passed": not failures, "failures": failures, "warnings": warnings,
               "contrasts": contrast_rows})
if failures:
    detail = "\n".join(f"  [{i + 1}] {m}" for i, m in enumerate(failures))
    raise AssertionError(f"NB 12 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 12 gate: PASSED")

## Notes carried forward

- **`e4_contrasts.csv` is the file that decides the novelty claim**, not the pooled MAE ordering.
  A lower mean with a confidence interval spanning zero is not a difference, and Table 4 should
  report the contrast rather than inviting the reader to subtract two column entries.
- **Report `localization_quality.json` beside every E4 number.** With fallback near zero, E4g's
  heuristic-box floor is what bounds the learned localizer's contribution — not fallback
  contamination.
- **If E4d fails**, the honest write-up is: two independent predictors (this one and NB 05's
  frozen encoder) found no benefit from per-lung decomposition; the paper's contribution is the
  structured evidence and auditability, not the decomposition. Protocol risk K2 anticipated this
  and the framing is already prepared.
- **E4f remains unavailable.** Without ground-truth boxes a null cannot be attributed between
  localization and decomposition. No new radiologist-scored set is available, so state this
  limitation explicitly and use E4g only as a heuristic floor.
- `anatomy_aware_predictions.jsonl` uses the shared schema, so NB 13's registry consumes the
  winning arm as agent A2's anatomy-aware variant without special-casing.